# Wo verliert der Nahverkehr gegen das Auto?

**Starter-Notebook für Projekt 1 in Informatik und Nachhaltigkeit (WiSe 2026/27)**

Dieses Notebook fragt den Routenplaner [MOTIS](https://github.com/motis-project/motis) ab,
der auf Ihrem Rechner läuft: erst eine einzelne Verbindung mit Bus und Bahn, mit dem Auto
und mit dem Rad, dann viele Startpunkte in Dortmund. Am Ende steht dieselbe Karte wie in
der [Webdemo](https://ls11ae.github.io/iun-routenplanung/reisezeit.html), nur gröber.

**Vorher:**
1. Datenpaket holen: `python daten/herunterladen.py`
2. MOTIS starten: im Ordner `nahverkehr` die Datei `starten.bat` doppelklicken (Windows) oder
   `./starten.sh` ausführen (macOS, Linux). Beim ersten Mal lädt das Skript MOTIS (52 MB) und
   bereitet die Daten vor (etwa eine Minute, gut 1 GB Arbeitsspeicher). Das Fenster offen lassen.
   Unter <http://localhost:8080> hat MOTIS auch eine eigene Oberfläche.

**Daten:** Fahrplan von [gtfs.de](https://gtfs.de) (DELFI e.V., CC BY 4.0), Wege aus
OpenStreetMap (© OpenStreetMap-Mitwirkende, ODbL).

In [1]:
import json
import statistics
import warnings
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

import pandas as pd

import reisezeitkarte as rk

try:
    print("Verbunden mit", rk.server_pruefen())
except Exception as fehler:
    raise SystemExit("MOTIS läuft nicht. Bitte zuerst nahverkehr/starten.bat (oder ./starten.sh) starten.") from fehler

Verbunden mit MOTIS 2.11.4


## 1. Gilt der Fahrplan noch?

Der Feed von gtfs.de deckt nur etwa einen Monat ab. Gerechnet wird für den nächsten
Dienstag, der im Fahrplan liegt. Zeiten gibt man MOTIS in Weltzeit (UTC); die Funktionen
in `reisezeitkarte.py` rechnen aus der Ortszeit um.

In [2]:
stand = json.loads((rk.WURZEL / "daten" / "stand.json").read_text(encoding="utf-8"))["gtfs"]
von, bis = (datetime.strptime(stand[k], "%Y%m%d").date() for k in ("gueltig_von", "gueltig_bis"))
print(f"Fahrplan gültig vom {von:%d.%m.%Y} bis {bis:%d.%m.%Y}.")
if date.today() > bis:
    warnings.warn("Der Fahrplan ist abgelaufen: git pull, python daten/herunterladen.py, dann MOTIS neu starten.")

beginn = max(von, date.today())
TAG = beginn + timedelta(days=(1 - beginn.weekday()) % 7)          # 1 = Dienstag
if TAG > bis:
    TAG = bis - timedelta(days=(bis.weekday() - 1) % 7)
ABFAHRT = rk.zeitpunkt(TAG.isoformat(), "07:30")                   # eine einzelne Abfahrt (Ortszeit)
ZEITEN = rk.abfahrten(rk.zeitpunkt(TAG.isoformat(), "07:00"), rk.zeitpunkt(TAG.isoformat(), "08:00"), 5)
wochentag = ["Montag", "Dienstag", "Mittwoch", "Donnerstag", "Freitag", "Samstag", "Sonntag"][TAG.weekday()]
print(f"Start: {wochentag}, {ABFAHRT:%d.%m.%Y, %H:%M} Uhr | in UTC: {rk.als_utc(ABFAHRT)}")

Fahrplan gültig vom 07.11.2026 bis 07.12.2026.
Start: Dienstag, 17.11.2026, 07:30 Uhr | in UTC: 2026-11-17T06:30:00Z


## 2. Eine Anfrage

Von Dortmund-Hörde zum Campus Nord. `rk.verbindungen` ruft die Schnittstelle
`/api/v6/plan` auf und liefert die Verbindungen mit ihren Abschnitten (*legs*).

In [3]:
START = (51.4862, 7.5050)      # Dortmund-Hörde (Breite, Länge)
ZIEL = rk.ZIEL                 # TU Dortmund, Campus Nord

ortszeit = lambda text: datetime.fromisoformat(text.replace("Z", "+00:00")).astimezone(ZoneInfo("Europe/Berlin"))

def als_tabelle(verbindungen):
    return pd.DataFrame([{
        "ab": ortszeit(v["startTime"]).strftime("%H:%M"), "an": ortszeit(v["endTime"]).strftime("%H:%M"),
        "Dauer (min)": round(v["duration"] / 60), "Umstiege": v["transfers"],
        "Weg": " → ".join(l.get("routeShortName") or ("zu Fuß" if l["mode"] == "WALK" else l["mode"]) for l in v["legs"]),
    } for v in verbindungen])

verbindungen = rk.verbindungen(START, ZIEL, ABFAHRT, timetableView="false")
als_tabelle(verbindungen)

,ab,an,Dauer (min),Umstiege,Weg
0,07:33,08:09,36,0,zu Fuß → 445 → zu Fuß


Wie lange dauert der Weg? Das hängt davon ab, wann man losgeht: Wer den Bus knapp verpasst,
wartet. `rk.oepnv_ueber_abfahrten` rechnet deshalb für jede Abfahrt zwischen 7:00 und
8:00 Uhr im Abstand von 5 Minuten die Zeit vom Verlassen des Hauses bis zur frühesten
Ankunft (die Wartezeit zählt also mit) und bildet den Median. Dafür genügt eine einzige
Anfrage über das ganze Zeitfenster.

In [4]:
fuss = rk.strassenzeiten([START], ZIEL, "WALK")[0]
oepnv = rk.oepnv_ueber_abfahrten(START, ZIEL, ZEITEN, fuss)
pd.DataFrame({"Abfahrt": [t.strftime("%H:%M") for t in ZEITEN], "Reisezeit (min)": oepnv["je_abfahrt"]}).set_index("Abfahrt").T

Abfahrt,07:00,07:05,07:10,07:15,07:20,07:25,07:30,07:35,07:40,07:45,07:50,07:55,08:00
Reisezeit (min),37.0,43.0,42.0,37.0,39.0,40.0,39.0,44.0,39.0,37.0,39.0,40.0,37.0


Im Vergleich mit Auto, Rad und Fußweg (`rk.strassenzeiten`). Der Campus ist autofrei; das
Auto fährt deshalb bis zur Emil-Figge-Straße (`rk.ZIEL_AUTO`).

In [5]:
pd.Series({
    "Bus und Bahn, Median 7:00–8:00 (min)": oepnv["min"],
    "… je nach Abfahrtsminute (min)": f"{oepnv['beste_min']:.0f} bis {oepnv['schlechteste_min']:.0f}",
    "… Fußweg zum ersten Halt (min)": oepnv["zum_halt_min"],
    "Bus und Bahn, schnellste Verbindung ohne Warten (min)": oepnv["dauer_min"],
    "Auto, reine Fahrzeit (min)": rk.strassenzeiten([START], rk.ZIEL_AUTO, "CAR")[0],
    "Fahrrad (min)": rk.strassenzeiten([START], ZIEL, "BIKE")[0],
    "zu Fuß (min)": fuss,
}, name="Hörde → Campus Nord").to_frame()

,Hörde → Campus Nord
"Bus und Bahn, Median 7:00–8:00 (min)",39.0
… je nach Abfahrtsminute (min),37 bis 44
… Fußweg zum ersten Halt (min),8.0
"Bus und Bahn, schnellste Verbindung ohne Warten (min)",33.0
"Auto, reine Fahrzeit (min)",10.8
Fahrrad (min),32.4
zu Fuß (min),106.4


## 3. Die Verbindung auf der Karte

Mit `detailedLegs="true"` enthält jeder Abschnitt seinen Verlauf als kodierte Linie
(*polyline*). Fußwege sind gepunktet, Fahrten durchgezogen gezeichnet.

In [6]:
import folium

def linie_dekodieren(text, genauigkeit=6):
    """Kodierte Linie (Google-Polyline-Verfahren) in eine Liste (Breite, Länge)."""
    punkte, i, lat, lon = [], 0, 0, 0
    while i < len(text):
        for achse in (0, 1):
            ergebnis, verschiebung = 0, 0
            while True:
                b = ord(text[i]) - 63
                i += 1
                ergebnis |= (b & 0x1F) << verschiebung
                verschiebung += 5
                if b < 0x20:
                    break
            d = ~(ergebnis >> 1) if ergebnis & 1 else ergebnis >> 1
            if achse == 0:
                lat += d
            else:
                lon += d
        punkte.append((lat / 10 ** genauigkeit, lon / 10 ** genauigkeit))
    return punkte

beste = min(rk.verbindungen(START, ZIEL, ABFAHRT, timetableView="false", detailedLegs="true"), key=lambda v: v["endTime"])
karte = folium.Map(location=START, zoom_start=13, tiles="OpenStreetMap")
for l in beste["legs"]:
    g = l["legGeometry"]
    zu_fuss = l["mode"] == "WALK"
    folium.PolyLine(linie_dekodieren(g["points"], g.get("precision", 6)), color="#555555" if zu_fuss else "#0072B2",
                    weight=4 if zu_fuss else 6, dash_array="1 8" if zu_fuss else None,
                    tooltip="zu Fuß" if zu_fuss else l.get("routeShortName", l["mode"])).add_to(karte)
folium.Marker(START, tooltip="Start").add_to(karte)
folium.Marker(ZIEL, tooltip="Ziel", icon=folium.Icon(color="black", icon="flag")).add_to(karte)
karte

## 4. Viele Startpunkte

`rk.raster_erzeugen` legt ein Gitter über das Stadtgebiet; `rk.reisezeiten_berechnen`
fragt für jeden Punkt Bus und Bahn (Median über die Abfahrten), Auto, Rad und Fußweg ab.
`rk.haltestellen_abstand` ergänzt die Luftlinie zur nächsten Haltestelle, an der zwischen
7 und 9 Uhr etwas abfährt. Hier mit 1000 m Maschenweite, damit es schnell geht (etwa eine
Minute). Die Karte der Webdemo hat 500 m: `python nahverkehr/reisezeitkarte.py` (etwa fünf Minuten).

In [7]:
RASTER = 1000
punkte = rk.raster_erzeugen(rk.grenze_laden(), RASTER)
zellen = rk.reisezeiten_berechnen(punkte, ZEITEN)
zellen["halt_m"] = rk.haltestellen_abstand(punkte, TAG)
t = pd.DataFrame(zellen)
t.describe().loc[["count", "min", "50%", "max"]].round(1)

Zu Fuß, Rad, Auto: 283 Punkte in 7.6 s


  ÖPNV:   100 von 283 Punkten,    20 s


  ÖPNV:   200 von 283 Punkten,    35 s


  ÖPNV:   283 von 283 Punkten,    47 s


ÖPNV: 283 Punkte in 47 s, Fehler: 0


,lat,lon,oepnv_min,oepnv_beste_min,oepnv_schlechteste_min,oepnv_dauer_min,umstiege,zum_halt_min,auto_min,rad_min,fuss_min,halt_m
count,283.0,283.0,280.0,280.0,280.0,280.0,280.0,276.0,273.0,280.0,280.0,283.0
min,51.4,7.3,7.8,7.8,7.8,7.8,-1.0,1.0,1.3,2.5,7.8,7.0
50%,51.5,7.5,52.0,44.0,59.0,42.0,1.0,13.0,12.6,39.5,132.4,278.0
max,51.6,7.6,90.0,77.0,105.0,77.0,3.0,42.0,24.1,74.7,252.9,1028.0


## 5. Das Verhältnis und die Karte

Wie viel länger dauert es mit Bus und Bahn als mit dem Auto? Die Autozeit aus
OpenStreetMap ist ohne Stau gerechnet. `ZUSCHLAG` sind Minuten für Parkplatzsuche und den
Fußweg vom Parkplatz (Standard wie in der Webdemo: 5); probieren Sie auch 0 und 10.
`rk.kategorie` ordnet jede Zelle ein. Zwei Fälle sind eigene Kategorien:
der Campus ist zu Fuß in höchstens 15 Minuten erreichbar, oder es gibt keine Haltestelle
mit Abfahrt zwischen 7 und 9 Uhr im Umkreis von 1 km.

In [8]:
from abbildung_reisezeit import RAMPE, SONDER, klassennamen    # dieselben Klassen und Farben wie in der Abbildung

ZUSCHLAG = 5
NAMEN = klassennamen(rk.VERHAELTNIS_GRENZEN)
SONDERNAMEN = {"campus": "Campus zu Fuß in höchstens 15 min", "kein_halt": "keine Haltestelle in 1 km",
               "kein_wert": "kein Wert"}
t["kategorie"] = [rk.kategorie(zellen, i, ZUSCHLAG) for i in range(len(t))]

dlon, dlat = rk.zelle_in_grad(ZIEL, RASTER)
karte = folium.Map(location=(51.515, 7.465), zoom_start=11, tiles="OpenStreetMap")
for z in t.itertuples():
    ecken = [(z.lat - dlat / 2, z.lon - dlon / 2), (z.lat + dlat / 2, z.lon + dlon / 2)]
    if isinstance(z.kategorie, str):
        folium.Rectangle(ecken, color=SONDER[z.kategorie]["edgecolor"], weight=1.5, dash_array="4 3", fill=True,
                         fill_color=SONDER[z.kategorie]["facecolor"], fill_opacity=0.6, tooltip=SONDERNAMEN[z.kategorie]).add_to(karte)
    else:
        v = z.oepnv_min / (z.auto_min + ZUSCHLAG)
        folium.Rectangle(ecken, color="#ffffff", weight=0.5, fill=True, fill_color=RAMPE[z.kategorie], fill_opacity=0.8,
                         tooltip=f"Bus und Bahn {z.oepnv_min:.0f} min, Auto {z.auto_min + ZUSCHLAG:.0f} min: {v:.1f}-mal so lange").add_to(karte)
folium.Marker(ZIEL, tooltip="Campus Nord", icon=folium.Icon(color="black", icon="flag")).add_to(karte)
karte

In [9]:
# Die Karte als Tabelle: Anteil der Zellen je Klasse, für verschiedene Zuschläge
def anteile(zuschlag):
    k = pd.Series([rk.kategorie(zellen, i, zuschlag) for i in range(len(t))])
    mit = k[k.apply(lambda x: isinstance(x, int))]
    v = [zellen["oepnv_min"][i] / (zellen["auto_min"][i] + zuschlag) for i in mit.index]
    zeile = {NAMEN[j] + " so lange": (mit == j).mean() for j in range(len(NAMEN))}
    return {**zeile, "Median des Verhältnisses": pd.Series(v).median(), "Zellen mit Verhältnis": len(mit)}

tabelle = pd.DataFrame({f"Auto + {z} min": anteile(z) for z in (0, 5, 10)})
tabelle.style.format("{:.0%}").format("{:.1f}", subset=pd.IndexSlice[["Median des Verhältnisses"], :]).format("{:.0f}", subset=pd.IndexSlice[["Zellen mit Verhältnis"], :])

,Auto + 0 min,Auto + 5 min,Auto + 10 min
"höchstens 2,5-mal so lange",0%,16%,74%
"2,5- bis 3-mal so lange",4%,33%,21%
"3- bis 3,5-mal so lange",13%,35%,5%
"3,5- bis 4-mal so lange",20%,11%,1%
mehr als 4-mal so lange,64%,4%,0%
Median des Verhältnisses,4.3,3.0,2.3
Zellen mit Verhältnis,268,268,268


**Lesehilfe:** Dunkle Zellen heißen nicht „schlechter Nahverkehr“, sondern „großes
Verhältnis“. Nah am Ziel ist das Auto in zwei Minuten da; mit dem Bus dauert es dann schnell
fünfmal so lange, obwohl es nur um wenige Minuten geht. Deshalb sind Zellen in Gehweite zum
Campus eine eigene Kategorie, und der Zuschlag für das Parken ist Standard. Das Raster ist
außerdem gleichmäßig über die Fläche gelegt: Felder und Industriegebiete zählen so viel wie
Wohnviertel.

## 6. Ihr Projekt: Ideen

1. **Annahmen prüfen und begründen:** Parkplatzsuche und Stau (Zuschlag je Stadtteil oder
   Tageszeit?), Startzeit und Takt (Median über eine Stunde, schlechteste Abfahrt oder
   schnellste Verbindung?), Gehweite zur Haltestelle (hier 1 km Luftlinie), Verspätungen.
2. **Für wen?** Gewichten Sie die Zellen mit Einwohnerzahlen (Zensus 2022, 100-m-Raster).
   Und: Wer hat überhaupt ein Auto?
3. **Andere Ziele und Zeiten:** Innenstadt, Klinikum, abends, am Sonntag. Wie stabil ist das Bild?
4. **Das Rad:** als eigenes Verkehrsmittel (`rad_min`) und als Zubringer zur S-Bahn.
5. **Wo ist der Engpass, und was ändert eine einzelne Maßnahme?** Dafür ist es praktisch,
   den Fahrplan selbst in der Hand zu haben: siehe `fahrplan_selbst_rechnen.ipynb`.
6. **Barrierefrei:** MOTIS kennt ein Rollstuhlprofil (`pedestrianProfile="WHEELCHAIR"` als
   weiterer Parameter von `rk.verbindungen`). Wie ändert sich die Karte?

Bitte nennen Sie auf Ihrem Poster die Quellen: Fahrplan gtfs.de / DELFI e.V. (CC BY 4.0),
Karte © OpenStreetMap-Mitwirkende (ODbL), berechnet mit MOTIS.